# Flink WordCount - Jupyter Notebook 示例

本 Notebook 演示如何使用 PyFlink 进行词频统计。

In [ ]:
# 安装 PyFlink
!pip install apache-flink==1.18.0

In [ ]:
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment, EnvironmentSettings

# 创建执行环境
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

# 创建 Table 环境
settings = EnvironmentSettings.new_instance()\
    .in_streaming_mode()\
    .build()
table_env = StreamTableEnvironment.create(env, settings)

In [ ]:
# 创建示例数据
data = [
    "apache flink is a framework",
    "flink is great for stream processing",
    "stream processing is fun",
    "apache flink rocks"
]

# 创建 DataStream
ds = env.from_collection(data)

In [ ]:
from pyflink.datastream.functions import FlatMapFunction
from pyflink.common.typeinfo import Types

# 定义分词函数
class Tokenizer(FlatMapFunction):
    def flat_map(self, value, collector):
        for word in value.lower().split():
            collector.collect((word, 1))

# 执行 WordCount
result = ds\
    .flat_map(Tokenizer(), output_type=Types.TUPLE([Types.STRING(), Types.INT()]))\
    .key_by(lambda x: x[0])\
    .sum(1)

# 打印结果
result.print()
env.execute("WordCount")

## Table API 版本

In [ ]:
# 使用 Table API
table_env.execute_sql("""
    CREATE TEMPORARY TABLE word_source (
        line STRING
    ) WITH (
        'connector' = 'datagen',
        'rows-per-second' = '1'
    )
""")

result_table = table_env.sql_query("""
    SELECT word, COUNT(*) as count
    FROM word_source,
    LATERAL TABLE(SPLIT(line, ' ')) AS T(word)
    GROUP BY word
""")

table_env.to_data_stream(result_table).print()
env.execute("Table WordCount")

## 练习

1. 修改代码过滤掉长度小于3的单词
2. 尝试使用 Sliding Window 统计每5秒的单词数
3. 将结果写入 Kafka